# MINI Cells — Experiment 019: Stable Proposal Utility Discovery

Official corrected rerun after the original `e=0` gradient-oracle numerical failure. The scientific design is unchanged; only the forward-equivalent gated-replicator autodiff numerics are stabilized. Phase-1 and donor models are checkpointed so oracle/features can be remeasured without retraining.


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys
ROOT = Path('/kaggle/working/mini-cells')
os.chdir('/kaggle/working')
if not (ROOT / '.git').exists():
    if ROOT.exists(): shutil.rmtree(ROOT)
    subprocess.run(['git','clone','--depth','1','https://github.com/ArcheLabs/mini-cells.git',str(ROOT)], check=True)
else:
    subprocess.run(['git','fetch','origin'], cwd=ROOT, check=True)
    subprocess.run(['git','switch','main'], cwd=ROOT, check=True)
    subprocess.run(['git','reset','--hard','origin/main'], cwd=ROOT, check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[lm]'], cwd=ROOT, check=True)
os.chdir(ROOT)
print('repo:', ROOT)
subprocess.run(['git','rev-parse','HEAD'], check=True)


In [ ]:
import torch
print({'python': sys.version.split()[0], 'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu_count': torch.cuda.device_count(), 'gpus': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]})
if torch.cuda.device_count() < 1:
    raise RuntimeError('Experiment 019 requires a Kaggle GPU accelerator')


## Preflight invariants

This gate covers the proposal-utility semantics, the reproduced zero-gate numerical pathology, the stable forward-equivalent fix, checkpoint round-trips, finite postprocessing, and the reused 018b/017/016 mechanisms.


In [ ]:
tests = [
    'tests/research/02-self-organization/test_language_recruitment_numerics.py',
    'tests/research/02-self-organization/test_language_proposal_checkpoints.py',
    'tests/research/02-self-organization/test_language_proposal_utility.py',
    'tests/research/02-self-organization/test_proposal_utility_resumable.py',
    'tests/research/02-self-organization/test_language_pressure_recruitment.py',
    'tests/research/02-self-organization/test_language_localized_learning.py',
    'tests/research/02-self-organization/test_language_growing_organism.py',
]
subprocess.run([sys.executable,'-m','pytest',*tests,'-q'], cwd=ROOT, check=True)


## Corrected checkpointed 3-replicate run

The first run trains Phase-1 plus six donor tissues and one RANDOM control per replicate. It writes 24 Kaggle-local checkpoints. Re-running the same command restores those checkpoints and only remeasures oracle/features. Use `--postprocess-only` only when stable worker CSV/JSON files are already complete and no model remeasurement is needed.


In [ ]:
OUT = ROOT / 'results' / 'proposal-utility-discovery-stable-v1'
subprocess.run([sys.executable,'scripts/research/run_proposal_utility_discovery_stable.py'], cwd=ROOT, check=True)
print('results:', OUT)


In [ ]:
import json, pandas as pd
from IPython.display import Image, Markdown, display
decision = json.loads((OUT / 'decision.json').read_text(encoding='utf-8'))
manifest = json.loads((OUT / 'checkpoint-manifest.json').read_text(encoding='utf-8'))
display(Markdown(f"## {decision['status']}"))
display(Markdown(f"**Question:** {decision['question']}"))
display(pd.DataFrame([decision['results']]))
display(pd.DataFrame([manifest]))
display(pd.read_csv(OUT / 'finite-audit.csv'))
display(pd.read_csv(OUT / 'oracle-consistency.csv'))
display(pd.read_csv(OUT / 'donor-summary.csv').head(30))
display(pd.read_csv(OUT / 'estimator-results.csv'))
for name in ['oracle-gradient-vs-fd.png','candidate-utility-matrix.png','heldout-spearman.png','heldout-auc.png','heldout-top1.png','heldout-regret.png','feature-oracle-correlations.png']:
    path = OUT / name
    if path.exists(): display(Image(filename=str(path)))


## Curate and publish

Stable results publish to a separate branch/artifact path so the original invalid 019 run remains auditable. Model checkpoints stay Kaggle-local and are not pushed to GitHub. Set `MINICELLS_PUBLISH=0` to prepare without pushing.


In [ ]:
publish = os.environ.get('MINICELLS_PUBLISH', '1').strip().lower() not in {'0','false','no'}
cmd = [sys.executable, 'scripts/research/publish_experiment_019_stable_results.py']
if publish: cmd.append('--push')
subprocess.run(cmd, cwd=ROOT, check=True)
